This notebook generates the mining views of the paper from the dataset obtained with the SQL query over MIMIC-III v1.4 ([Initial-query/query.txt](../Initial-query/query.txt)).

Input: `../datasets/mimic-iii-dataset.csv`. Outputs:
- `../datasets/mining-view-clinical.csv`: main analysis (clinical cultures).
- `../datasets/mining-view-all-specimens.csv`: sensitivity analysis (clinical cultures and swabs).
- `../datasets/auxiliary-data-clinical.csv` and `../datasets/auxiliary-data-all-specimens.csv`: identifiers and flags aligned by row with the mining views (not used for mining).

IMPORTANT: these files contain MIMIC-III data and must not be distributed. No individual record is displayed in this notebook.

In [1]:
import platform
import pandas as pd
import numpy as np
import sklearn
import matplotlib
from sklearn.preprocessing import LabelEncoder
print("Python " + platform.python_version() + " | pandas " + pd.__version__ + " | numpy " + np.__version__ + " | scikit-learn " + sklearn.__version__ + " | matplotlib " + matplotlib.__version__)

Python 3.13.16 | pandas 3.0.5 | numpy 2.5.3 | scikit-learn 1.9.1 | matplotlib 3.11.2


# 1. Reading and describing the dataset

In [2]:
mimic_iii_dataset_path = "../datasets/mimic-iii-dataset.csv"
df = pd.read_csv(mimic_iii_dataset_path)
print("Number of instances: " + str(len(df)))
print("Number of attributes: " + str(len(df.columns)))

Number of instances: 12796
Number of attributes: 44


In [3]:
df.isnull().sum()

patient_id                                                            0
patient_gender                                                        0
patient_date_of_birth                                                 0
patient_age                                                           0
admission_id                                                          0
admission_date                                                        0
admission_date_of_the_previous_admission                           8580
readmission                                                           0
exitus                                                                0
admission_type                                                        0
admission_location                                                    0
discharge_location                                                    0
admission_date_in_the_first_icu                                     135
time_between_admission_and_first_icu                            

In [4]:
df["culture_susceptibility"].value_counts(dropna=False)

culture_susceptibility
S    10608
R     2140
I       48
Name: count, dtype: int64

# 2. Selection of the cohort

## 2.1. Enterococcus isolates

The query returns all the vancomycin-tested isolates (mainly Enterococcus, *S. aureus* and coagulase-negative staphylococci). Since the species mix is different in each unit, a spatial effect could be only a species effect. Therefore, we only keep the Enterococcus isolates and the target is the vancomycin susceptibility (R vs. S).

In [5]:
pd.crosstab(df["culture_microorganism_name"].str.startswith("ENTEROCOCCUS"), df["culture_susceptibility"])

culture_susceptibility,I,R,S
culture_microorganism_name,,,
False,14,9,8114
True,34,2131,2494


In [6]:
df = df[df["culture_microorganism_name"].str.startswith("ENTEROCOCCUS")].copy()
print("Number of instances: " + str(len(df)))

Number of instances: 4659


## 2.2. Adult patients

Neonates are a different population (neonatal ICU, specific microbiology). We delete the NEWBORN admissions, the patients younger than 16 years and the isolates whose ICU is 'NICU' or 'NWARD' (neonatal units).

In [7]:
df = df[(df["admission_type"] != "NEWBORN") & (df["patient_age"] >= 16) & (~df["icu_when_culture"].isin(["NICU", "NWARD"]))].copy()
print("Number of instances: " + str(len(df)))

Number of instances: 4649


## 2.3. Clinical cultures and swabs

The swabs (rectal screening for VRE, in most cases) are processed in a selective medium, so they are almost always resistant, and the screening policy is different in each unit. To avoid learning the screening policy instead of the resistance, the main analysis only uses clinical cultures. The swabs are kept in the sensitivity analysis.

In [8]:
screening_specimens = ["SWAB", "MRSA SCREEN", "Staph aureus Screen"]
df["is_swab"] = df["culture_specimen_type_description"].isin(screening_specimens)
pd.crosstab([df["is_swab"], df["icu_when_culture"]], df["culture_susceptibility"])

culture_susceptibility     I    R    S
is_swab icu_when_culture              
False   CCU                0   55  200
        CSRU               0   98  112
        MICU               6  464  568
        NO_ICU            12  289  658
        SICU               1  340  387
        TSICU              1   95  254
True    CCU                0    9    6
        CSRU               0   30    6
        MICU               2  116   36
        NO_ICU             1   66   69
        SICU               9  502  140
        TSICU              2   67   48

## 2.4. First isolate per patient and admission

Deleting duplicated rows after deleting the identifiers would merge different patients with the same profile and would not guarantee one isolate per admission. Therefore, we keep the first Enterococcus isolate (genus level, since many isolates are only identified as 'ENTEROCOCCUS SP.') per patient and admission, using the identifiers and the culture date.

In [9]:
df["culture_date"] = pd.to_datetime(df["culture_date"])
df["admission_date"] = pd.to_datetime(df["admission_date"])
df["admission_date_in_the_first_icu"] = pd.to_datetime(df["admission_date_in_the_first_icu"])
df = df.sort_values(["culture_date", "isolate_number", "culture_microorganism_name"], kind="mergesort")
df_clinical = df[~df["is_swab"]].drop_duplicates(subset=["patient_id", "admission_id"], keep="first").reset_index(drop=True)
df_all_specimens = df.drop_duplicates(subset=["patient_id", "admission_id"], keep="first").reset_index(drop=True)
print("Clinical cultures (main analysis): " + str(len(df_clinical)) + " isolates, " + str(df_clinical["patient_id"].nunique()) + " patients.")
print("All specimens (sensitivity analysis): " + str(len(df_all_specimens)) + " isolates, " + str(df_all_specimens["patient_id"].nunique()) + " patients.")

Clinical cultures (main analysis): 2505 isolates, 2343 patients.
All specimens (sensitivity analysis): 3119 isolates, 2873 patients.


# 3. Cleaning and transforming data

We apply the same transformations to both views, with a function. The cut points of the discretizations are computed with the main view and reused in the other one.

## 3.1. Attribute 'hours_from_admission_to_first_icu'

The attribute 'time_between_admission_and_first_ICU' of the query also counts the ICU stays that started after the culture (information from the future). We recompute it with the dates: if the first ICU stay started after the culture (or there is no ICU stay), the value is 'NO_ICU_BEFORE_CULTURE'. Negative values are fixed to 0 and the positive values are discretized with the 33rd and 66th percentiles.

In [10]:
def hours_to_first_icu(dataset):
    hours = (dataset["admission_date_in_the_first_icu"] - dataset["admission_date"]) / pd.Timedelta(hours=1)
    hours = hours.clip(lower=0).fillna(-1).astype(int)
    before_culture = dataset["admission_date_in_the_first_icu"].notna() & (dataset["admission_date_in_the_first_icu"] <= dataset["culture_date"])
    return hours.where(before_culture, -1)

In [11]:
hours = hours_to_first_icu(df_clinical)
print("ICU stays after the culture or no ICU stay: " + str((hours == -1).sum()))
print("Consistency with 'icu_when_culture' = 'NO_ICU': " + str(((hours == -1) == (df_clinical["icu_when_culture"] == "NO_ICU")).mean()))
att_33th_percentile = np.percentile([x for x in hours if x > 0], 33)
att_66th_percentile = np.percentile([x for x in hours if x > 0], 66)
print("33rd percentile: " + str(att_33th_percentile))
print("66th percentile: " + str(att_66th_percentile))

ICU stays after the culture or no ICU stay: 759
Consistency with 'icu_when_culture' = 'NO_ICU': 1.0
33rd percentile: 15.0
66th percentile: 83.68


In [12]:
def discretize_hours(hours):
    labels = {0 : "NO_ICU_BEFORE_CULTURE", 1 : "0", 2 : "0<x<=" + str(int(att_33th_percentile)), 3 : str(int(att_33th_percentile)) + "<x<=" + str(int(att_66th_percentile)), 4 : str(int(att_66th_percentile)) + "<x"}
    return pd.Series(np.digitize(hours, [-0.5, 0.0, att_33th_percentile, att_66th_percentile], right=True)).map(labels)
discretize_hours(hours).value_counts()

0                        1047
NO_ICU_BEFORE_CULTURE     759
83<x                      238
0<x<=15                   234
15<x<=83                  227
Name: count, dtype: int64

## 3.2. Other attributes

- 'patient_age': 'ADULT' [16, 65) and 'ELDERLY' [65, 90] (no children after the selection of the cohort).
- 'admission_type': 'URGENT' is merged with 'EMERGENCY'.
- 'admission_location': grouped into EMERGENCY_ROOM, TRANSFER_FROM_HOSPITAL, TRANSFER_FROM_OTHER_FACILITY, REFERRAL and NO_INFO.
- 'culture_specimen_type': grouped specimen types.
- 'culture_susceptibility': 'I' is considered 'R'.
- 'service_when_culture': services with few isolates are grouped into 'OTHER'.
- 'previous_colonization_with_*' is renamed to 'previous_isolation_of_*': the query checks the presence of the microorganism in previous admissions, not its resistance (e.g., 'previous_colonization_with_erv' is any Enterococcus).
- Deleted attributes: identifiers and dates; 'exitus' and 'discharge_location' (known after the culture); 'culture_month' (no temporal component); 'culture_microorganism_name' (all the isolates are Enterococcus); 'stay_in_icu_in_previous_admissions' and 'previous_exposure_to_glycopeptides' (highly correlated with other attributes).

In [13]:
admission_location_groups = {
    "EMERGENCY ROOM ADMIT" : "EMERGENCY_ROOM",
    "TRANSFER FROM HOSP/EXTRAM" : "TRANSFER_FROM_HOSPITAL",
    "TRANSFER FROM SKILLED NUR" : "TRANSFER_FROM_OTHER_FACILITY",
    "TRANSFER FROM OTHER HEALT" : "TRANSFER_FROM_OTHER_FACILITY",
    "TRSF WITHIN THIS FACILITY" : "TRANSFER_FROM_OTHER_FACILITY",
    "CLINIC REFERRAL/PREMATURE" : "REFERRAL",
    "PHYS REFERRAL/NORMAL DELI" : "REFERRAL",
    "HMO REFERRAL/SICK" : "REFERRAL",
    "** INFO NOT AVAILABLE **" : "NO_INFO",
}
specimen_groups = {
    "URINE" : "URINE", "URINE,KIDNEY" : "URINE",
    "BLOOD CULTURE" : "BLOOD", "BLOOD CULTURE ( MYCO/F LYTIC BOTTLE)" : "BLOOD", "FLUID RECEIVED IN BLOOD CULTURE BOTTLES" : "BLOOD",
    "PERITONEAL FLUID" : "ABDOMINAL_FLUID", "BILE" : "ABDOMINAL_FLUID",
    "ABSCESS" : "ABSCESS_TISSUE_WOUND", "TISSUE" : "ABSCESS_TISSUE_WOUND", "FLUID WOUND" : "ABSCESS_TISSUE_WOUND", "BIOPSY" : "ABSCESS_TISSUE_WOUND",
    "FOOT CULTURE" : "ABSCESS_TISSUE_WOUND", "FOREIGN BODY" : "ABSCESS_TISSUE_WOUND", "ASPIRATE" : "ABSCESS_TISSUE_WOUND",
    "CATHETER TIP-IV" : "CATHETER_TIP",
    "SWAB" : "SWAB", "MRSA SCREEN" : "SWAB", "Staph aureus Screen" : "SWAB",
}
main_services = ["MED", "SURG", "CMED", "NMED", "CSURG", "TRAUM", "NSURG", "OMED", "VSURG", "TSURG", "NO_SERVICE"]

In [14]:
def transform(dataset):
    result = pd.DataFrame(index=dataset.index)
    result["patient_gender"] = dataset["patient_gender"]
    result["patient_age"] = np.where(dataset["patient_age"] >= 65, "ELDERLY", "ADULT")
    result["readmission"] = dataset["readmission"]
    result["admission_type"] = dataset["admission_type"].replace({"URGENT" : "EMERGENCY"})
    result["admission_location"] = dataset["admission_location"].map(admission_location_groups).fillna("NO_INFO")
    result["culture_specimen_type"] = dataset["culture_specimen_type_description"].map(specimen_groups).fillna("OTHER")
    for column in dataset.columns:
        if column.startswith("treated_with_vancomycin") or column.startswith("previous_exposure_to") or column.endswith("in_previous_admissions"):
            result[column] = dataset[column]
        elif column.startswith("previous_colonization_with_"):
            result[column.replace("previous_colonization_with_", "previous_isolation_of_")] = dataset[column]
    result = result.drop(columns=["previous_exposure_to_glycopeptides", "stay_in_icu_in_previous_admissions"], errors="ignore")
    # Context attributes.
    result["icu_when_culture"] = dataset["icu_when_culture"]
    result["service_when_culture"] = dataset["service_when_culture"].where(dataset["service_when_culture"].isin(main_services), "OTHER")
    result["hours_from_admission_to_first_icu"] = discretize_hours(hours_to_first_icu(dataset)).values
    # Target.
    result["culture_susceptibility"] = dataset["culture_susceptibility"].replace({"I" : "R"})
    # The subgroups library does not support spaces or commas in the values.
    result = result.astype(str).apply(lambda column : column.str.replace(" ", "_").str.replace(",", "_"))
    return result.reset_index(drop=True)

In [15]:
def auxiliary_data(dataset):
    result = pd.DataFrame({
        "patient_id" : dataset["patient_id"].values,
        "admission_id" : dataset["admission_id"].values,
        "culture_date" : dataset["culture_date"].values,
        # A culture at exactly 00:00:00 almost always comes from CHARTDATE (only the date is known).
        "culture_date_only" : (dataset["culture_date"].dt.strftime("%H:%M:%S") == "00:00:00").values,
        "enterococcus_species" : dataset["culture_microorganism_name"].values,
        "intermediate" : (dataset["culture_susceptibility"] == "I").values,
        "culture_specimen_type_description" : dataset["culture_specimen_type_description"].values,
        "hours_from_admission_to_first_icu_raw" : hours_to_first_icu(dataset).values,
    })
    return result

In [16]:
mining_view_clinical = transform(df_clinical)
mining_view_all_specimens = transform(df_all_specimens)
auxiliary_clinical = auxiliary_data(df_clinical)
auxiliary_all_specimens = auxiliary_data(df_all_specimens)
print("Clinical view: " + str(mining_view_clinical.shape))
print("All-specimens view: " + str(mining_view_all_specimens.shape))

Clinical view: (2505, 28)
All-specimens view: (3119, 28)


## 3.3. Deletion of attributes with only one value

In [17]:
single_value = [c for c in mining_view_clinical.columns if mining_view_clinical[c].nunique() < 2]
print("Attributes with only one value: " + str(single_value))
mining_view_clinical = mining_view_clinical.drop(columns=single_value)
mining_view_all_specimens = mining_view_all_specimens.drop(columns=single_value)

Attributes with only one value: []


# 4. Correlation between attributes

In [18]:
numeric_df = mining_view_clinical.copy()
le = LabelEncoder()
for c in numeric_df.columns:
    numeric_df[c] = le.fit_transform(numeric_df[c])
corr_matrix = numeric_df.corr().abs()
for i in range(len(corr_matrix.columns) - 1):
    for j in range(i + 1, len(corr_matrix.columns)):
        if corr_matrix.iloc[i, j] >= 0.8:
            print("Attributes '" + corr_matrix.columns[i] + "' and '" + corr_matrix.columns[j] + "' are correlated (" + str(round(corr_matrix.iloc[i, j], 2)) + ").")

'icu_when_culture' and 'hours_from_admission_to_first_icu' share the value for the patients without ICU before the culture ('NO_ICU' / 'NO_ICU_BEFORE_CULTURE'). Both are context attributes and are kept: the first one is the unit and the second one is the trajectory until the first ICU.

# 5. Summarization and description of data

In [19]:
print("Number of columns: " + str(len(mining_view_clinical.columns)))
print("Number of rows (clinical): " + str(len(mining_view_clinical)))
print("Number of rows (all specimens): " + str(len(mining_view_all_specimens)))

Number of columns: 28
Number of rows (clinical): 2505
Number of rows (all specimens): 3119


In [20]:
# Values of the columns (main view).
for c in mining_view_clinical.columns:
    print("** COLUMN " + c.upper() + " **")
    print(mining_view_clinical[c].value_counts(dropna=False).to_string())

** COLUMN PATIENT_GENDER **
patient_gender
M    1267
F    1238
** COLUMN PATIENT_AGE **
patient_age
ELDERLY    1461
ADULT      1044
** COLUMN READMISSION **
readmission
no     1728
yes     777
** COLUMN ADMISSION_TYPE **
admission_type
EMERGENCY    2306
ELECTIVE      199
** COLUMN ADMISSION_LOCATION **
admission_location
EMERGENCY_ROOM                  1178
REFERRAL                         763
TRANSFER_FROM_HOSPITAL           513
TRANSFER_FROM_OTHER_FACILITY      50
NO_INFO                            1
** COLUMN CULTURE_SPECIMEN_TYPE **
culture_specimen_type
URINE                   1500
BLOOD                    512
ABSCESS_TISSUE_WOUND     182
ABDOMINAL_FLUID          156
OTHER                    109
CATHETER_TIP              46
** COLUMN TREATED_WITH_VANCOMYCIN_IN_PREVIOUS_ADMISSIONS **
treated_with_vancomycin_in_previous_admissions
no     2029
yes     476
** COLUMN TREATED_WITH_VANCOMYCIN_IN_THE_CURRENT_ADMISSION_BEFORE_CULTURE **
treated_with_vancomycin_in_the_current_admission_befo

In [21]:
pd.crosstab(mining_view_clinical["icu_when_culture"], mining_view_clinical["culture_susceptibility"], margins=True)

culture_susceptibility,R,S,All
icu_when_culture,,,
CCU,37,171,208
CSRU,61,84,145
MICU,315,416,731
NO_ICU,213,546,759
SICU,151,272,423
TSICU,52,187,239
All,829,1676,2505


# 6. Write the mining views

In [22]:
mining_view_clinical.to_csv("../datasets/mining-view-clinical.csv", index=False)
mining_view_all_specimens.to_csv("../datasets/mining-view-all-specimens.csv", index=False)
auxiliary_clinical.to_csv("../datasets/auxiliary-data-clinical.csv", index=False)
auxiliary_all_specimens.to_csv("../datasets/auxiliary-data-all-specimens.csv", index=False)